In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 50.9 MB/s eta 0:00:00:00:0100:01
Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [3]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
predictions = zs(prompt_150, labels_150)
predictions, ans_150

({'sequence': 'Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.',
  'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."',
   'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."',
   'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."',
   'The butterfly effect is the phenomenon that a small change in the init

In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

prompt_emb = model.encode([prompt_150])
D, I = index.search(prompt_emb, 10)

retrieved_indices = I[0]

true_rank = None
for rank, idx in enumerate(retrieved_indices, start=1):
    if idx == 150:
        true_rank = rank
        break

print("Q2 =", true_rank)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Q2 = 10


In [6]:
docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]

ce_scores = cross_encoder.predict(pairs)

order = np.argsort(-ce_scores)

ce_rank = None
for r, pos in enumerate(order, start=1):
    if retrieved_indices[pos] == 150:
        ce_rank = r
        break

print("Q3 =", ce_rank)

Q3 = 1


In [7]:
row42 = train.iloc[42]
prompt42 = str(row42["prompt"])

emb42 = model.encode([prompt42])
_, idx42 = index.search(emb42, 5)

docs5 = [kb[i] for i in idx42[0]]

rag_string = "Context: " + " ".join(docs5) + " Question: " + prompt42

tokens = tokenizer(rag_string, truncation=False)

print("Q4 =", len(tokens["input_ids"]))

Q4 = 216


In [8]:
true_doc = kb[150]

rag = "Context: " + true_doc + " Question: " + prompt_150

result = zs(rag, labels_150)

score = 0

for label, s in zip(result["labels"], result["scores"]):
    if label == ans_150:
        score = s

print("Q5 =", round(score,3))

Q5 = 0.989


Bad RAG

In [9]:
bad_doc = kb[999]

adv = "Context: " + bad_doc + " Question: " + prompt_150

result = zs(adv, labels_150)

score = 0

for label, s in zip(result["labels"], result["scores"]):
    if label == ans_150:
        score = s

print("Q6 =", round(score,3))

Q6 = 0.529


In [10]:
hits = 0

for i in range(100):

    row = train.iloc[i]

    prompt = str(row["prompt"])
    correct = str(row[row["answer"]])

    emb = model.encode([prompt])
    _, idx = index.search(emb,5)

    docs = [kb[j] for j in idx[0]]

    if any(correct in d for d in docs):
        hits += 1

hit_rate = hits/100*100

print("Q7 =", round(hit_rate,1))

Q7 = 73.0


In [11]:
def map3(actual, predicted):
    score = 0.0

    for k,p in enumerate(predicted[:3], start=1):
        if p==actual:
            score = 1.0/k
            break

    return score

scores=[]

for i in range(20):

    row = train.iloc[i]

    prompt = str(row["prompt"])

    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    answer_letter = row["answer"]

    emb = model.encode([prompt])
    _, idx = index.search(emb,5)

    docs = [kb[j] for j in idx[0]]

    pairs = [[prompt,d] for d in docs]

    ce = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce)]

    rag = "Context: " + best_doc + " Question: " + prompt

    result = zs(rag, labels)

    label_to_letter = {
        str(row["A"]):"A",
        str(row["B"]):"B",
        str(row["C"]):"C",
        str(row["D"]):"D",
        str(row["E"]):"E",
    }

    predicted_letters = [label_to_letter[x] for x in result["labels"][:3]]

    scores.append(map3(answer_letter,predicted_letters))

print("Q8 =", round(np.mean(scores),3))

Q8 = 0.975
